In [3]:
import hail as hl
import os
import pandas as pd
from ukb_utils import hail_init
from ukb_utils import genotypes
from ukb_utils import samples
from ukb_utils import variants
from ko_utils import qc
from ko_utils import analysis

In [4]:
os.chdir('/well/lindgren/UKBIOBANK/flassen/projects/KO/wes_ko_ukbb/')
hail_init.hail_bmrc_init('logs/hail/hail_test_export.log', 'GRCh38')

Running on Apache Spark version 3.1.2
SparkUI available at http://compa010.hpc.in.bmrc.ox.ac.uk:4040
Welcome to
     __  __     <>__
    / /_/ /__  __/ /
   / __  / _ `/ / /
  /_/ /_/\_,_/_/_/   version 0.2.77-684f32d73643
LOGGING: writing to logs/hail/hail_test_export.log


In [44]:
PATH_GENE_INTERVALS = 'derived/tables/gene_intervals/211111_wes200k_saige_wes_saige_sig_genes_intervals.txt'
PHENOTYPE = 'breast_cancer'

In [45]:
ht = hl.import_table(PATH_GENE_INTERVALS, force_bgz = True, delimiter = '\t', missing = '',
                     types={'contig': hl.tstr,'start': hl.tint, 'end': hl.tint, 'contig_length': hl.tint}) 
ht = ht.annotate(contig = hl.delimit(['chr',ht.contig],''))


2021-11-28 21:00:22 Hail: INFO: Reading table without type imputation
  Loading field 'gene' as type str (not specified)
  Loading field 'FDR' as type str (not specified)
  Loading field 'maf' as type str (not specified)
  Loading field 'mutation' as type str (not specified)
  Loading field 'phenotype' as type str (not specified)
  Loading field 'contig' as type str (user-supplied)
  Loading field 'start' as type int32 (user-supplied)
  Loading field 'end' as type int32 (user-supplied)
  Loading field 'contig_length' as type int32 (user-supplied)


In [46]:
padding = 500000
ht = ht.annotate(ranges = hl.array([ht.start,ht.end]))
ht = ht.annotate(
    end = hl.max(ht.ranges),
    start = hl.min(ht.ranges))
ht = ht.annotate(
    end_with_padding = hl.min(hl.array([ht.end + padding, ht.contig_length])),
    start_with_padding = hl.max(hl.array([ht.start - padding, 1])))
ht = ht.drop(ht.ranges)
ht.head(5).show()

,,,,,,,,,,
gene,FDR,maf,mutation,phenotype,contig,start,end,contig_length,end_with_padding,start_with_padding
str,str,str,str,str,str,int32,int32,int32,int32,int32
"""ENSG00000136297""","""0.0795317615353971""","""00_01""","""ptv""","""coronary_artery_disease""","""chr7""",4905989,4959213,159345973,5459213,4405989
"""ENSG00000188611""","""0.0795317615353971""","""00_01""","""ptv""","""coronary_artery_disease""","""chr10""",50182778,50279720,133797422,50779720,49682778
"""ENSG00000105501""","""0""","""00_01""","""ptv""","""chronic_obstructive_pulmonary_disease""","""chr19""",51611927,51645545,58617616,52145545,51111927
"""ENSG00000178460""","""0""","""00_01""","""ptv""","""chronic_obstructive_pulmonary_disease""","""chr8""",66870749,66922048,145138636,67422048,66370749
"""ENSG00000187546""","""0""","""00_01""","""ptv""","""chronic_obstructive_pulmonary_disease""","""chr7""",15200317,15562015,159345973,16062015,14700317


In [47]:
reference_genome = 'GRCh38'
ht = ht.annotate(start_valid = hl.is_valid_locus(ht.contig, ht.start, reference_genome))
ht = ht.annotate(end_valid = hl.is_valid_locus(ht.contig, ht.end, reference_genome))
ht = ht.annotate(valid_intervals = ht.start_valid & ht.end_valid)
defined_coords = hl.is_defined(ht.valid_intervals)
n_drop = ht.filter(~defined_coords).count()
if n_drop > 0:
    print(f'Dropping {n_drop} undefined start/end genomic coordinates.')
    ht = ht.filter(defined_coords)
invalid_intervals = ht.filter(~ht.valid_intervals).count()
assert invalid_intervals == 0, 'Some of the supplied intervals are outside current chromosome contig'
ht = ht.annotate(intervals = hl.locus_interval(ht.contig, ht.start, ht.end, True, True, reference_genome=reference_genome))


Dropping 1 undefined start/end genomic coordinates.


In [48]:
ht = ht.filter(ht.phenotype == PHENOTYPE)

In [49]:
ht.show()

,,,,,,,,,,,,,,
gene,FDR,maf,mutation,phenotype,contig,start,end,contig_length,end_with_padding,start_with_padding,start_valid,end_valid,valid_intervals,intervals
str,str,str,str,str,str,int32,int32,int32,int32,int32,bool,bool,bool,interval<locus<GRCh38>>
"""ENSG00000168280""","""0""","""00_01""","""ptv_damaging_missense""","""breast_cancer""","""chr2""",148875227,149026759,242193529,149526759,148375227,True,True,True,[chr2:148875227-chr2:149026759]
"""ENSG00000179348""","""0""","""00_01""","""ptv_damaging_missense""","""breast_cancer""","""chr3""",128479427,128493201,198295559,128993201,127979427,True,True,True,[chr3:128479427-chr3:128493201]
"""ENSG00000054356""","""0""","""00_01""","""synonymous""","""breast_cancer""","""chr2""",219289623,219309648,242193529,219809648,218789623,True,True,True,[chr2:219289623-chr2:219309648]
"""ENSG00000103375""","""0""","""00_01""","""synonymous""","""breast_cancer""","""chr16""",25215731,25228932,90338345,25728932,24715731,True,True,True,[chr16:25215731-chr16:25228932]
"""ENSG00000150477""","""0""","""00_01""","""synonymous""","""breast_cancer""","""chr18""",36829106,37232172,80373285,37732172,36329106,True,True,True,[chr18:36829106-chr18:37232172]
"""ENSG00000164897""","""0""","""00_01""","""synonymous""","""breast_cancer""","""chr7""",151081085,151083493,159345973,151583493,150581085,True,True,True,[chr7:151081085-chr7:151083493]
"""ENSG00000174944""","""0.0829886563419236""","""00_01""","""synonymous""","""breast_cancer""","""chr3""",151212117,151278542,198295559,151778542,150712117,True,True,True,[chr3:151212117-chr3:151278542]
"""ENSG00000176428""","""0""","""00_01""","""synonymous""","""breast_cancer""","""chr7""",73667831,73672112,159345973,74172112,73167831,True,True,True,[chr7:73667831-chr7:73672112]


In [56]:
chromosomes = [x.replace('chr','') for x in list(set(ht.contig.collect()))]
chromosomes = ['2']

In [57]:
mt = genotypes.get_ukb_genotypes_bed(chromosomes)
mt = mt.annotate_rows(**{'info': hl.agg.call_stats(mt.GT, mt.alleles)})

2021-11-28 21:01:50 Hail: INFO: Found 488377 samples in fam file.
2021-11-28 21:01:50 Hail: INFO: Found 61966 variants in bim file.


In [58]:
# filter to common variants
mt = mt.annotate_rows(info = mt.info.annotate(AC = mt.info.AC[1]))
mt = mt.annotate_rows(info = mt.info.annotate(AF = mt.info.AF[1]))
mt = qc.filter_min_maf(mt, 0.01)

In [62]:
# perform liftover to GRCh38
mt = variants.liftover(mt)

In [66]:
# filter to variants in the the locus
mt = hl.filter_intervals(mt, ht.intervals.collect())

In [ ]:
# count variants in locus
n = mt.count()
print('filtered')